# 02 · Análisis de variables y dinámica provincial

**Objetivo:** estudiar la información adicional de los indicadores económicos, demográficos y turísticos respecto a la historia del precio.

Se estudian evolución, heterogeneidad, asociaciones en niveles y crecimientos, retardos adicionales, redundancia, estabilidad e información mutua. Cada apartado presenta las evidencias y su interpretación. El cuaderno no cambia modelos, hiperparámetros ni la selección congelada.

**Separación de usos:** la evolución de toda la muestra es descriptiva. Las correlaciones, transformaciones exploratorias y diagnósticos que motivan variables emplean solo desarrollo, con orígenes y etiquetas hasta 2018Q4. Consultar resultados recientes aquí no convierte la evaluación en un test ciego.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
ROOT = Path.cwd()
if not (ROOT / 'sources.json').is_file():
    ROOT = ROOT.parent
assert (ROOT / 'sources.json').is_file(), 'Abrir el cuaderno desde el repositorio o notebooks/'
REPORTS = ROOT / 'data/reports/unified'
EDA = ROOT / 'data/reports/eda'
pd.set_option('display.max_columns', 20)

In [ ]:
import numpy as np
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from forecasting.direct import build_design, FEATURE_BLOCKS, EXTERNAL_BLOCKS
from forecasting.experiment import SELECTION_CUTOFF
panel = pd.read_parquet(ROOT / 'data/processed/panel_provincial_trimestral.parquet')
panel['quarter'] = pd.PeriodIndex(panel.quarter, freq='Q')
panel['province_code'] = panel.province_code.astype(str).str.zfill(2)
cutoff = pd.Period(SELECTION_CUTOFF)
dev = panel.loc[panel.quarter.le(cutoff)].copy()
corr = pd.read_csv(EDA / 'correlaciones_crecimiento.csv')
levels = pd.read_csv(EDA / 'correlaciones_nivel.csv')
stability = pd.read_csv(EDA / 'correlaciones_estabilidad.csv')
coverage = pd.read_csv(EDA / 'cobertura_variables.csv')
external = sorted({v for block in EXTERNAL_BLOCKS.values() for v in block})
design4 = build_design(dev, 4)
pairs4 = design4.loc[design4.target_quarter.le(cutoff)
                    & design4.current_price.gt(0) & design4.target_price.gt(0)].copy()
assert pairs4.target_quarter.max() <= cutoff

## 1. Evolución del nivel y del crecimiento: descripción de toda la muestra

Un nivel alto no implica un crecimiento alto. La primera gráfica muestra ambas dimensiones y la dispersión provincial. Las medianas y percentiles son resúmenes de territorios sin ponderación por población o transacciones; no representan un índice nacional.

El área entre percentiles territoriales describe heterogeneidad observada. **No es un intervalo predictivo.** Las etapas históricas sirven para contextualizar, no constituyen una prueba formal de cambio estructural.

In [ ]:
display(Image(filename=str(EDA / 'figuras/evolucion_precio_crecimiento.png')))
evolution = pd.read_csv(EDA / 'evolucion_anual.csv')
display(evolution.tail(12))
print('Este apartado incluye datos posteriores a desarrollo y es exclusivamente descriptivo.')

## 2. Diferencias territoriales al mismo punto de partida

Se compara el cierre de desarrollo, no el último dato de evaluación. El gráfico ordena territorios por valor tasado y la tabla muestra también su variación interanual. La identidad provincial puede explicar parte de las diferencias de nivel; eso no garantiza anticipar cambios temporales.

Predecir correctamente que una provincia tiene precios más altos que otra puede producir un buen ajuste agregado y, a la vez, errores relevantes en el crecimiento de ambas.

In [ ]:
snapshot = dev.loc[dev.quarter.eq(cutoff), ['province', 'valor_tasado_m2', 'valor_tasado_m2_yoy']]
snapshot = snapshot.dropna(subset=['valor_tasado_m2']).sort_values('valor_tasado_m2')
display(pd.concat([snapshot.head(8), snapshot.tail(8)]))
selected = pd.concat([snapshot.head(8), snapshot.tail(8)]).set_index('province')
selected.valor_tasado_m2.plot.barh(figsize=(10, 6), color='#2878B5')
plt.title(f'Ocho valores menores y mayores en {cutoff}: comparación territorial')
plt.xlabel('Valor tasado (€/m²)'); plt.tight_layout(); plt.show()

## 3. Por qué se incluyen variables externas

Los bloques representan dimensiones distintas: crédito y operaciones, economía, demografía, turismo y oferta. El IPC mide inflación general; no deflacta el objetivo. El euríbor representa condiciones financieras y pertenece al bloque de crédito. Los indicadores turísticos se normalizan por población para distinguir intensidad de tamaño territorial.

Estas son motivaciones para contrastar predictores, no relaciones causales ya demostradas. La tabla indica las variables realmente incluidas en la parrilla; otras columnas del panel pueden quedar fuera por cobertura o diseño.

In [ ]:
display(pd.DataFrame([{'bloque': k, 'variables': ', '.join(v), 'n_externas': len(v)}
                      for k, v in EXTERNAL_BLOCKS.items()]))
display(coverage.loc[coverage.variable.isin(external),
                    ['variable', 'primera_observacion', 'cobertura_desarrollo_pct',
                     'cobertura_train_inicial_pct']].sort_values('cobertura_desarrollo_pct'))

## 4. Correlación con el nivel frente a correlación con crecimiento futuro

Spearman mide asociación monotónica, no mejora de un pronóstico. La correlación contemporánea con el nivel puede reflejar diferencias provinciales o tendencias compartidas. La asociación con `100 × log(P[t+4]/P[t])` permite estudiar la relación entre el indicador conocido en t y el crecimiento posterior.

Comparar estas dos columnas no equivale a una prueba estadística de diferencia: las muestras disponibles pueden variar. Por eso se muestran los números de pares y no solo los coeficientes.

In [ ]:
level_table = levels.loc[levels.variable.isin(external), ['variable', 'n_pares', 'spearman_agrupada']]
annual = corr.loc[corr.horizonte.eq(4) & corr.variable.isin(external)]
comparison = level_table.merge(annual[['variable', 'n_pares', 'spearman_agrupada']],
    on='variable', suffixes=('_nivel', '_crecimiento_futuro'))
display(comparison.sort_values('spearman_agrupada_crecimiento_futuro', key=abs, ascending=False))
focus = ['ipc_yoy', 'euribor_12m', 'tasa_paro', 'pib_per_capita_eur',
         'viajeros_hoteles_por_1000_hab', 'hipotecas_numero_por_1000_hab']
comparison.set_index('variable').reindex(focus)[
    ['spearman_agrupada_nivel', 'spearman_agrupada_crecimiento_futuro']].plot.barh(figsize=(11, 5))
plt.axvline(0, color='grey', lw=.8); plt.xlim(-1, 1)
plt.title('Asociación con nivel y crecimiento anual futuro: desarrollo')
plt.tight_layout(); plt.show()

## 5. Asociación agrupada y asociación tras centrar por provincia

El informe resta las medias provinciales de X e Y en la muestra de pares y aplica Spearman. Esto permite comprobar cuánto cambia la asociación al retirar diferencias medias territoriales. No es una regresión causal de efectos fijos, no elimina tendencias comunes y no resuelve la dependencia entre observaciones.

Una diferencia grande entre ambas asociaciones invita a investigar si el patrón es territorial o temporal; no decide automáticamente si se debe conservar la variable.

In [ ]:
centered = annual[['variable', 'n_pares', 'spearman_agrupada', 'spearman_sin_media_provincial']].copy()
centered['cambio_al_centrar'] = centered.spearman_sin_media_provincial - centered.spearman_agrupada
display(centered.sort_values('cambio_al_centrar', key=abs, ascending=False))
display(Image(filename=str(EDA / 'figuras/correlaciones_crecimiento.png')))

## 6. IPC, euríbor, paro y turismo: inspección de las nubes de puntos

Las nubes se construyen con indicadores del origen y etiquetas anuales ya observables dentro de desarrollo. El crecimiento se mide en puntos logarítmicos porcentuales, aproximadamente porcentajes para cambios pequeños. La línea horizontal solo marca crecimiento nulo.

La dispersión permite detectar relaciones débiles, concentraciones y extremos que un coeficiente resume en exceso. No se dibuja una recta causal ni se interpretan las miles de filas como observaciones independientes.

In [ ]:
variables = ['ipc_yoy', 'euribor_12m', 'tasa_paro', 'viajeros_hoteles_por_1000_hab']
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, variable in zip(axes.flat, variables):
    sample = pairs4[[variable, 'target_log_growth']].dropna()
    ax.scatter(sample[variable], sample.target_log_growth, s=7, alpha=.18, rasterized=True)
    ax.axhline(0, color='grey', lw=.8)
    ax.set(title=f'{variable}: {len(sample)} pares', xlabel=variable,
           ylabel='Crecimiento futuro a 4T (100 × log)')
plt.tight_layout(); plt.show()

## 7. La asociación depende del horizonte

Una variable puede estar asociada con el crecimiento del próximo trimestre y tener poca relación con el de tres años. Las etiquetas de todos los horizontes deben terminar antes del corte de desarrollo; por ello sus últimos orígenes y tamaños muestrales difieren.

El mapa usa una escala fija de −1 a 1. No se escoge el horizonte de mayor correlación después de mirar el test: estos son diagnósticos exploratorios de desarrollo.

In [ ]:
focus = ['ipc_yoy', 'euribor_12m', 'tasa_paro', 'pib_per_capita_eur',
         'renta_neta_media_persona', 'hipotecas_numero_por_1000_hab',
         'viajeros_hoteles_por_1000_hab', 'pernoctaciones_hoteles_por_1000_hab',
         'pct_poblacion_18_34', 'iniciadas_libre_por_1000_hab']
by_horizon = corr.loc[corr.variable.isin(focus)].pivot(
    index='variable', columns='horizonte', values='spearman_sin_media_provincial').reindex(focus)
display(by_horizon)
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(by_horizon.to_numpy(), aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_yticks(range(len(by_horizon)), by_horizon.index)
ax.set_xticks(range(len(by_horizon.columns)), by_horizon.columns)
ax.set(xlabel='Horizonte (trimestres)', title='Spearman tras centrar por provincia: desarrollo')
fig.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

## 8. Retardos adicionales de indicadores externos

La disponibilidad ya está retrasada en el panel. El retardo adicional de 0, 1 o 4 trimestres se aplica **encima** de esa regla: no permite acceder a publicaciones futuras. Se consulta el trimestre exacto y los huecos permanecen ausentes.

Este apartado comprueba si cambia la asociación al mirar información más antigua. No incorpora automáticamente esos retardos a los modelos seleccionados. Elegir el máximo de muchas correlaciones generaría optimismo y requeriría otra validación.

In [ ]:
lagged = pd.read_csv(EDA / 'correlaciones_retardos_externos.csv')
lag_view = lagged.loc[lagged.horizonte.eq(4) & lagged.variable.isin(focus)]
display(lag_view.pivot(index='variable', columns='retardo_adicional',
                       values='spearman_sin_media_provincial'))
display(lag_view[['variable', 'retardo_adicional', 'n_pares']])

## 9. Estabilidad histórica: una media puede ocultar cambios

Las ventanas 1995–2007, 2008–2013 y 2014–2018 son descriptivas y están prefijadas. Tanto origen como objetivo deben estar dentro de cada ventana. Se comparan signo, magnitud y tamaño de muestra.

Un cambio de signo es una señal para investigar, no una prueba formal de ruptura. En particular, una celda vacía o con pocos pares no permite concluir ausencia de relación. Ninguna ventana de este análisis llega al periodo de evaluación reciente.

In [ ]:
stable = stability.loc[stability.horizonte.eq(4) & stability.variable.isin(focus)]
signs = stable.pivot(index='variable', columns='regimen', values='spearman_sin_media_provincial')
sizes = stable.pivot(index='variable', columns='regimen', values='n_pares')
display(signs); display(sizes)
flips = signs.apply(lambda row: bool((row.dropna() > 0).any() and (row.dropna() < 0).any()), axis=1)
print('Variables con signos distintos entre ventanas disponibles:', ', '.join(flips.index[flips]) or 'ninguna')

## 10. Redundancia: añadir columnas puede repetir información

Las pernoctaciones y los viajeros, o ciertas medidas de población, pueden compartir mucha información. La tabla de redundancia muestra pares con asociación elevada en desarrollo; el mapa compara un conjunto concreto de predictores sobre los datos disponibles.

Se usan pares completos, por lo que no todos los coeficientes comparten la misma muestra. La regularización puede ayudar cuando existen predictores relacionados, pero no elimina el problema de cobertura ni garantiza una mejora fuera de muestra. No se eliminan columnas automáticamente por superar un umbral.

In [ ]:
redundancy = pd.read_csv(EDA / 'redundancia.csv')
display(redundancy.loc[redundancy.variable_a.isin(external) & redundancy.variable_b.isin(external)])
columns = ['ipc_yoy', 'euribor_12m', 'tasa_paro', 'pib_per_capita_eur',
           'viajeros_hoteles_por_1000_hab', 'pernoctaciones_hoteles_por_1000_hab',
           'hipotecas_numero_por_1000_hab', 'compraventas_por_1000_hab']
matrix = dev[columns].corr(method='spearman', min_periods=40)
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(matrix, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(columns)), columns, rotation=65, ha='right')
ax.set_yticks(range(len(columns)), columns)
ax.set_title('Redundancia en desarrollo; observaciones disponibles por pares')
fig.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

## 11. Información mutua: otro diagnóstico, no un selector definitivo

La información mutua busca dependencias que no tienen por qué ser monotónicas. Se estima univariadamente, dentro de desarrollo, con la semilla fijada y una imputación aprendida en las filas etiquetadas de ese ámbito. Se expresa en nats, no como porcentaje de precio explicado.

Una puntuación alta puede reflejar patrones de ausencia o tiempo compartido, además de señal económica. La comparación con Spearman no decide qué modelo gana: ambos son análisis dentro de muestra y no miden directamente el error de predicción futuro.

In [ ]:
mi = pd.read_csv(EDA / 'informacion_mutua.csv')
mi4 = mi.loc[mi.horizonte.eq(4) & mi.variable.isin(external)].copy()
mi4['porcentaje_imputado'] = 100 * mi4.n_imputados / mi4.n_objetivos
display(mi4.sort_values('informacion_mutua_nats', ascending=False)[
    ['variable', 'informacion_mutua_nats', 'n_observados', 'porcentaje_imputado']])
top = mi4.nlargest(12, 'informacion_mutua_nats').sort_values('informacion_mutua_nats')
top.set_index('variable').informacion_mutua_nats.plot.barh(figsize=(10, 5), color='#2878B5')
plt.title('Información mutua con crecimiento anual: diagnóstico dentro de desarrollo')
plt.xlabel('Nats'); plt.tight_layout(); plt.show()

## 12. Sensibilidad a la muestra disponible

Dos correlaciones de variables distintas pueden haberse calculado sobre años y provincias diferentes. Aquí se repite la asociación anual de varios indicadores sobre una muestra común completa y se compara con la muestra disponible de cada uno. No se incluye renta en este ejemplo para evitar que su corta cobertura domine toda la comparación.

La muestra común mejora la comparabilidad, pero reduce la información y puede seleccionar periodos particulares. No resuelve todos los sesgos de ausencia. El resultado no cambia los bloques del benchmark.

In [ ]:
variables = ['ipc_yoy', 'euribor_12m', 'tasa_paro', 'viajeros_hoteles_por_1000_hab',
             'hipotecas_numero_por_1000_hab']
common = pairs4.dropna(subset=variables + ['target_log_growth'])
rows = []
for variable in variables:
    available = pairs4[[variable, 'target_log_growth']].dropna()
    rows.append({'variable': variable, 'n_disponible': len(available), 'n_común': len(common),
        'spearman_disponible': available[variable].corr(available.target_log_growth, method='spearman'),
        'spearman_muestra_común': common[variable].corr(common.target_log_growth, method='spearman')})
display(pd.DataFrame(rows))
print('Orígenes de la muestra común:', common.quarter.min(), 'a', common.quarter.max())

## 13. Síntesis por variable y paso a la prueba predictiva

La tabla combina cobertura, asociación anual centrada, cambios de signo históricos e información mutua. Permite formular hipótesis, pero no constituye un ranking automático definitivo. En particular, no prueba un efecto causal individual del IPC, de los tipos o del turismo.

El siguiente cuaderno compara el error al añadir cada bloque con el obtenido usando solo la historia del precio, bajo las mismas condiciones temporales. Una asociación alta no garantiza una reducción del error; una variable económicamente relevante puede no aportar una señal adicional utilizable con los datos disponibles.

In [ ]:
synthesis = annual[['variable', 'n_pares', 'spearman_sin_media_provincial']].merge(
    coverage[['variable', 'cobertura_desarrollo_pct']], on='variable', how='left')
synthesis = synthesis.merge(mi4[['variable', 'informacion_mutua_nats', 'porcentaje_imputado']],
                            on='variable', how='left')
synthesis['signos_distintos_entre_ventanas'] = synthesis.variable.map(flips)
display(synthesis.sort_values('variable'))
print('La inclusión efectiva se decide en selección temporal, no en esta tabla exploratoria.')

## 14. Conclusiones e interpretación del resultado del TFM

El análisis distingue nivel y crecimiento, diferencias provinciales y temporales, disponibilidad y asociación. Los indicadores económicos se estudian y prueban, aunque no todos lleguen a la previsión final. Su exclusión no demuestra que carezcan de influencia económica; mide el resultado de configuraciones y muestras concretas.

En el experimento guardado resultaron seleccionadas configuraciones con historia del precio y turismo o solo con historia del precio. Eso no convierte al turismo en una explicación causal ni garantiza una mejora en otra época. La evaluación posterior muestra que la selección no supera la tendencia interanual reciente; esa evidencia se desarrolla en el cuaderno 04, no se utiliza aquí para rediseñar retrospectivamente las variables.

**Límites:** dependencia entre provincias y fechas, tendencias compartidas, coberturas desiguales, pocas fechas de selección y ausencia de versiones históricas de publicaciones. Una ampliación futura debe formular y evaluar nuevas hipótesis con separación temporal, en lugar de seleccionar las correlaciones más favorables de la evaluación conocida.

**Siguiente cuaderno:** [03 · Selección de modelos y variables](03_seleccion_modelos_variables.ipynb).